# 03. Customer Segmentation (RFM Analysis)
**Proyek**: Solusi Prima Packaging Study Case  
**Peran**: Data Analyst  
**Tujuan**: Mengelompokkan pelanggan berdasarkan perilaku pembelian mereka untuk menentukan strategi pemasaran dan promosi yang tepat sasaran bagi reseller.

---

### Import Libraries & Load Cleaned Data

In [1]:
# %% [markdown]
# ### RFM Data Ingestion & Validation Pipeline Component

# %%
import os
import sys
import logging
import pandas as pd
from typing import Tuple

# =============================================================================
# 1. LOGGING & ENVIRONMENT CONFIGURATION
# =============================================================================
# Mengonfigurasi logging standar industri agar terintegrasi dengan monitoring sistem/cloud
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# Mengamankan base path secara absolut agar skrip bisa dijalankan dari direktori mana pun
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
PATH_PROCESSED = os.path.normpath(os.path.join(BASE_DIR, "../data/processed/"))

# Definisi skema data (Dtype Map) untuk mengoptimalkan memori RAM dan mengunci integritas tipe data
TRANSAKSI_DTYPE = {
    'order_id': 'string',
    'reseller_id': 'string',
    'customer_id': 'string',
    'product_id': 'string',
    'qty': 'int32',
    'harga': 'float64',
    'margin': 'float64'
}
PELANGGAN_DTYPE = {
    'customer_id': 'string',
    'nama_pelanggan': 'string',
    'jumlah_pembelian': 'int32',
    'feedback': 'string'
}

# =============================================================================
# 2. CORE DATA INGESTION FUNCTION (DEFENSIVE DESIGN)
# =============================================================================
def load_and_validate_rfm_data(
    base_path: str,
    transaksi_file: str = "cleaned_data_transaksi.csv",
    pelanggan_file: str = "cleaned_data_pelanggan.csv"
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Memuat data transaksi dan pelanggan disertai validasi integritas skema RFM.

    Args:
        base_path: Direktori absolut tempat berkas CSV disimpan.
        transaksi_file: Nama file berkas transaksi penjualan.
        pelanggan_file: Nama file berkas master data pelanggan.

    Returns:
        Tuple berisi:
        - df_transaksi (pd.DataFrame): Dataframe transaksi yang tervalidasi.
        - df_pelanggan (pd.DataFrame): Dataframe pelanggan yang tervalidasi.

    Raises:
        FileNotFoundError: Jika berkas target tidak ditemukan di jalur direktori.
        ValueError: Jika data kosong atau terdapat pelanggaran integritas kolom.
    """
    path_transaksi = os.path.join(base_path, transaksi_file)
    path_pelanggan = os.path.join(base_path, pelanggan_file)
    
    # Validasi eksistensi berkas secara proaktif sebelum dialokasikan ke memori (Fail-Fast)
    if not os.path.exists(path_transaksi):
        raise FileNotFoundError(f"Komponen Kritis Hilang: Berkas tidak ditemukan di {path_transaksi}")
    if not os.path.exists(path_pelanggan):
        raise FileNotFoundError(f"Komponen Kritis Hilang: Berkas tidak ditemukan di {path_pelanggan}")
        
    logger.info("Memulai proses memuat data untuk analisis segmentasi RFM...")
    
    # Ingestion dengan pemetaan tipe data statis
    df_transaksi = pd.read_csv(path_transaksi, dtype=TRANSAKSI_DTYPE)
    df_pelanggan = pd.read_csv(path_pelanggan, dtype=PELANGGAN_DTYPE)
    
    # Validasi anomali data kosong (Empty DataFrame Check)
    if df_transaksi.empty or df_pelanggan.empty:
        raise ValueError("Data Breach/Anomaly: Salah satu berkas CSV yang dimuat kosong.")
        
    # Validasi integritas kolom wajib untuk kalkulasi metrik Recency, Frequency, dan Monetary
    required_transaksi_cols = {'tanggal', 'customer_id', 'harga'}
    if not required_transaksi_cols.issubset(df_transaksi.columns):
        missing = required_transaksi_cols - set(df_transaksi.columns)
        raise ValueError(f"Skema Rusak: Kolom penting untuk RFM tidak ditemukan di transaksi: {missing}")
        
    # Standardisasi format tipe data tanggal dengan penanganan eror berbasis 'coerce'
    df_transaksi['tanggal'] = pd.to_datetime(df_transaksi['tanggal'], errors='coerce')
    
    # Penanganan rekaman tanggal yang korup demi akurasi kalkulasi komponen 'Recency'
    if df_transaksi['tanggal'].isnull().any():
        bad_rows = df_transaksi['tanggal'].isnull().sum()
        logger.warning(f"Data Quality Notice: Ditemukan {bad_rows} baris dengan format tanggal rusak (diabaikan dari pipeline).")
        df_transaksi = df_transaksi.dropna(subset=['tanggal'])
        
    # Pengecekan Integritas Referensial: Mendeteksi customer_id tak terdaftar di data transaksi
    unregistered_customers = set(df_transaksi['customer_id']) - set(df_pelanggan['customer_id'])
    if unregistered_customers:
        logger.warning(
            f"Referential Integrity Alert: Ditemukan {len(unregistered_customers)} customer_id "
            f"di data transaksi yang tidak terdaftar pada master data pelanggan."
        )
        
    logger.info(f"Ingestion Sukses. Transaksi: {df_transaksi.shape[0]} baris, Pelanggan: {df_pelanggan.shape[0]} baris.")
    return df_transaksi, df_pelanggan

# =============================================================================
# 3. PIPELINE EXECUTION GATEWAY
# =============================================================================
try:
    # Mengeksekusi fungsi pemuatan data secara aman
    df_transaksi, df_pelanggan = load_and_validate_rfm_data(base_path=PATH_PROCESSED)
    print("\n[SUCCESS] Data transaksi dan pelanggan siap diproses untuk analisis RFM.")
except Exception as e:
    logger.error(f"Pipeline Eksekusi RFM Ingestion Gagal: {str(e)}")
    # Melempar kembali eror agar orchestrator otomatisasi menyadari adanya kegagalan
    raise e

2026-06-22 21:40:24,696 [INFO] Memulai proses memuat data untuk analisis segmentasi RFM...
2026-06-22 21:40:24,712 [INFO] Ingestion Sukses. Transaksi: 100 baris, Pelanggan: 12 baris.

[SUCCESS] Data transaksi dan pelanggan siap diproses untuk analisis RFM.


### Menghitung Nilai RFM secara Programmatic

In [2]:
# %% [markdown]
# ### RFM Metrics Calculation Pipeline Component

# %%
from typing import Dict, Any

# =============================================================================
# 1. CORE RFM CALCULATION FUNCTION (OPTIMIZED & DEFENSIVE)
# =============================================================================
def calculate_rfm_metrics(df_transaksi_clean: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Menghitung metrik dasar Recency, Frequency, dan Monetary (RFM) per pelanggan.
    
    Menggunakan teknik Named Aggregation dan Vectorized Math untuk performa 
    eksekusi maksimal dan meminimalkan overhead memori.

    Args:
        df_transaksi_clean: Dataframe transaksi yang sudah bersih dan ter-parse datetimenya.

    Returns:
        Tuple berisi:
        - df_rfm_final (pd.DataFrame): Dataframe berisi metrik RFM per customer_id.
        - execution_meta (Dict): Metadata informasi acuan tanggal analisis untuk audit.

    Raises:
        ValueError: Jika dataframe input kosong atau kolom penting hilang.
    """
    # 1. Validasi Input Parameter (Fail-Fast Principle)
    required_cols = {'customer_id', 'tanggal', 'order_id', 'harga'}
    if df_transaksi_clean.empty:
        raise ValueError("Pipeline Error: Dataframe transaksi kosong. Tidak bisa memproses RFM.")
    if not required_cols.issubset(df_transaksi_clean.columns):
        missing = required_cols - set(df_transaksi_clean.columns)
        raise ValueError(f"Schema Error: Kolom berikut wajib ada untuk kalkulasi RFM: {missing}")

    logger.info("Memulai penghitungan metrik RFM secara vectorized...")

    # 2. Penentuan Tanggal Acuan Analisis (Snapshot Date) secara Defensif
    max_transaction_date = df_transaksi_clean['tanggal'].max()
    snapshot_date = max_transaction_date + pd.Timedelta(days=1)

    # 3. Eksekusi Agregasi dengan Named Aggregation (Aman & Menghindari Pergeseran Kolom)
    # Menghitung nilai transaksi agregasi dan tanggal maksimal terlebih dahulu
    df_rfm = df_transaksi_clean.groupby('customer_id').agg(
        last_transaction_date=('tanggal', 'max'),
        Frequency=('order_id', 'nunique'),
        Monetary=('harga', 'sum')
    ).reset_index()

    # 4. Optimasi Vectorized Math untuk Recency (Menghilangkan Lambda / Jauh Lebih Cepat)
    # Operasi pengurangan tanggal dilakukan langsung di level C-engine Pandas (Vectorized)
    df_rfm['Recency'] = (snapshot_date - df_rfm['last_transaction_date']).dt.days

    # 5. Restrukturisasi Struktur Kolom Akhir secara Eksplisit
    df_rfm_final = df_rfm[['customer_id', 'Recency', 'Frequency', 'Monetary']].copy()

    # Metadata untuk keperluan logging dan tracking audit sistem
    execution_meta = {
        "snapshot_date": snapshot_date.strftime('%Y-%m-%d'),
        "total_unique_customers": int(df_rfm_final['customer_id'].nunique()),
        "max_recency_days": int(df_rfm_final['Recency'].max()),
        "total_monetary_value": float(df_rfm_final['Monetary'].sum())
    }

    logger.info(
        f"Kalkulasi RFM Sukses. Berhasil memproses {execution_meta['total_unique_customers']:,} "
        f"pelanggan unik dengan basis tanggal acuan: {execution_meta['snapshot_date']}."
    )

    return df_rfm_final, execution_meta

# =============================================================================
# 2. PIPELINE EXECUTION GATEWAY
# =============================================================================
try:
    print("=== MENGHITUNG METRIK RFM PER PELANGGAN ===")

    # Mengeksekusi fungsi kalkulasi inti menggunakan dataframe dari cell ingestion sebelumnya
    df_rfm_metrics, rfm_meta = calculate_rfm_metrics(df_transaksi_clean=df_transaksi)

    # Menyimpan output ke variabel global 'rfm' agar selaras dengan kebutuhan cell downstream Anda
    rfm = df_rfm_metrics

    print("\nHasil Perhitungan Dasar RFM:")
    display(rfm.head())

except Exception as e:
    logger.error(f"Komponen Kalkulasi Metrik RFM Gagal: {str(e)}")
    raise e

=== MENGHITUNG METRIK RFM PER PELANGGAN ===
2026-06-22 21:40:24,746 [INFO] Memulai penghitungan metrik RFM secara vectorized...
2026-06-22 21:40:24,773 [INFO] Kalkulasi RFM Sukses. Berhasil memproses 12 pelanggan unik dengan basis tanggal acuan: 2025-08-19.

Hasil Perhitungan Dasar RFM:


,customer_id,Recency,Frequency,Monetary
0,CORP-01,2,10,56150000.0
1,CORP-02,8,7,77450000.0
2,CORP-03,1,14,90625000.0
3,CORP-04,1,9,29025000.0
4,CORP-05,4,10,76175000.0


### Lead Strategy: Scoring dan Penentuan Segmen

In [3]:
# %% [markdown]
# ### RFM Scoring & Customer Segmentation Pipeline Component

# %%
import numpy as np

OUTPUT_DIR = "../outputs/tables/"
OUTPUT_FILENAME = "customer_segmentation_rfm.csv"

# =============================================================================
# 2. CORE SCORING & SEGMENTATION FUNCTION (VECTORIZED)
# =============================================================================
def segment_customers_rfm(df_rfm: pd.DataFrame, df_pelanggan: pd.DataFrame) -> pd.DataFrame:
    """Melakukan scoring kuintil 3-skala dan segmentasi pelanggan secara vectorized.

    Args:
        df_rfm: Dataframe berisi metrik dasar Recency, Frequency, Monetary.
        df_pelanggan: Dataframe master katalog data pelanggan.

    Returns:
        pd.DataFrame: Dataframe rfm_final yang sudah ter-merge dengan segmen pasar.

    Raises:
        ValueError: Jika kolom metrik penting tidak ditemukan di dataframe input.
    """
    # Validasi awal integritas skema data (Fail-Fast Principle)
    required_cols = {'customer_id', 'Recency', 'Frequency', 'Monetary'}
    if not required_cols.issubset(df_rfm.columns):
        missing = required_cols - set(df_rfm.columns)
        raise ValueError(f"Schema Error: Kolom berikut wajib ada untuk proses scoring: {missing}")

    logger.info("Memulai proses scoring berbasis kuintil 3-skala...")
    
    # Mencegah mutasi pada dataframe asli cell sebelumnya
    rfm_working = df_rfm.copy()

    # Perhitungan Skor Kuintil menggunakan penanganan rank 'first' demi kestabilan distribusi data
    rfm_working['R_Score'] = pd.qcut(rfm_working['Recency'].rank(method='first'), q=3, labels=[3, 2, 1]).astype(int)
    rfm_working['F_Score'] = pd.qcut(rfm_working['Frequency'].rank(method='first'), q=3, labels=[1, 2, 3]).astype(int)
    rfm_working['M_Score'] = pd.qcut(rfm_working['Monetary'].rank(method='first'), q=3, labels=[1, 2, 3]).astype(int)

    logger.info("Mengeksekusi aturan segmentasi menggunakan metode vectorized arrays...")

    # OPTIMASI VECTORIZED: Mengganti fungsi baris-per-baris .apply() dengan np.select (Sangat Cepat)
    conditions = [
        (rfm_working['R_Score'] == 1),                                      # At Risk
        (rfm_working['R_Score'] == 3) & (rfm_working['F_Score'] == 3),      # Loyal
        (rfm_working['R_Score'] == 3) & (rfm_working['F_Score'] == 1)       # New Customer
    ]
    choices = ['At Risk', 'Loyal', 'New Customer']
    
    # Menetapkan nilai default 'Potential Loyalist' sebagai pengganti blok 'else' konvensional
    rfm_working['Segment'] = np.select(conditions, choices, default='Potential Loyalist')

    # Penggabungan data (Merge) secara defensif dengan master katalog pelanggan
    df_merged = rfm_working.merge(
        df_pelanggan[['customer_id', 'nama_pelanggan', 'feedback']], 
        on='customer_id', 
        how='left'
    )

    return df_merged


def export_segmentation_report(df_report: pd.DataFrame, output_dir: str, filename: str) -> None:
    """Menangani ekspor tabel laporan hasil segmentasi ke penyimpanan lokal secara aman."""
    try:
        os.makedirs(output_dir, exist_ok=True)
        full_export_path = os.path.join(output_dir, filename)
        
        # Ekspor berkas CSV bersih tanpa menyertakan indeks internal Pandas
        df_report.to_csv(full_export_path, index=False)
        logger.info(f"Artifact Laporan Sukses Diekspor ke: {full_export_path}")
    except PermissionError:
        logger.error(
            f"Permission Denied: Gagal menulis berkas ke {filename}. "
            "Pastikan file tersebut tidak sedang dibuka oleh aplikasi lain seperti Excel/Numbers."
        )
    except Exception as e:
        logger.error(f"Gagal mengekspor laporan segmentasi: {str(e)}")

# =============================================================================
# 3. PIPELINE EXECUTION GATEWAY
# =============================================================================
try:
    print("=== PROSES SCORING DAN SEGMENTASI ===")

    # Eksekusi fungsi inti kalkulasi segmentasi pasar
    rfm_final = segment_customers_rfm(df_rfm=rfm, df_pelanggan=df_pelanggan)

    print("\nHasil Akhir Segmentasi Pelanggan:")
    display(rfm_final[['customer_id', 'nama_pelanggan', 'Recency', 'Frequency', 'Monetary', 'Segment']])

    # Ekspor dataset laporan secara aman dan defensif
    export_segmentation_report(
        df_report=rfm_final,
        output_dir=OUTPUT_DIR,
        filename=OUTPUT_FILENAME
    )

except Exception as e:
    logger.error(f"Komponen Scoring & Segmentasi Pipeline Gagal: {str(e)}")
    raise e

=== PROSES SCORING DAN SEGMENTASI ===
2026-06-22 21:40:24,827 [INFO] Memulai proses scoring berbasis kuintil 3-skala...
2026-06-22 21:40:24,847 [INFO] Mengeksekusi aturan segmentasi menggunakan metode vectorized arrays...

Hasil Akhir Segmentasi Pelanggan:


,customer_id,nama_pelanggan,Recency,Frequency,Monetary,Segment
0,CORP-01,PT Makanan Sehat,2,10,56150000.0,Loyal
1,CORP-02,CV Minuman Segar,8,7,77450000.0,Potential Loyalist
2,CORP-03,UMKM Dapur Bunda,1,14,90625000.0,Loyal
3,CORP-04,Kopi Kita Bersama,1,9,29025000.0,Potential Loyalist
4,CORP-05,Pabrik Keripik Renyah,4,10,76175000.0,Potential Loyalist
5,CORP-06,PT Herbal Alami,37,7,64550000.0,At Risk
6,CORP-07,Brand Skincare Cantika,2,13,103625000.0,Potential Loyalist
7,CORP-08,Jus Buah Segaria,12,6,10850000.0,At Risk
8,CORP-09,Sambal Nona Roa,1,7,57550000.0,Potential Loyalist
9,CORP-10,PT Roti Enak,4,8,58175000.0,Potential Loyalist


2026-06-22 21:40:24,885 [INFO] Artifact Laporan Sukses Diekspor ke: ../outputs/tables/customer_segmentation_rfm.csv


### Visualisasi Distribusi Segmen Pelanggan

In [4]:
# %% [markdown]
# ### Customer Segmentation Visualization Pipeline Component

# %%
import plotly.graph_objects as go

OUTPUT_CHARTS_DIR = "../outputs/charts/"
OUTPUT_FILENAME_PNG = "customer_segments_distribution.png"

# McKinsey Tier Color Palette
C = {
    "navy":      "#0B1D35",             # Keperluan teks utama / kontras tinggi
    "tick":      "#6B7B8D",             # Angka sumbu, label ringan, dan pembatas
    "footnote":  "#9DA8B5",             # Teks keterangan bawah (low emphasis)
    "blue":      "#1558B0",             # Warna bar utama / segmen dominan
    "forecast":  "#2A7DE1",             # Elemen penunjang
    "slate":     "rgba(94,108,132,0.38)", # Elemen sekunder / bar pendukung
    "grid":      "rgba(11,29,53,0.065)",  # Garis pandu Y-axis (sangat tipis)
    "axis_line": "rgba(11,29,53,0.16)",   # Garis dasar sumbu X
    "bg":        "#FFFFFF"              # Latar belakang bersih (corporate white)
}

# =============================================================================
# 1. CORE VISUALIZATION FUNCTION
# =============================================================================
def generate_corporate_segment_chart(df_segmented: pd.DataFrame) -> go.Figure:
    """Membangun visualisasi distribusi segmen pelanggan bergaya McKinsey Ultra-Clean.

    Args:
        df_segmented: Dataframe hasil segmentasi akhir yang memuat kolom 'Segment'.

    Returns:
        plotly.graph_objects.Figure: Objek chart Plotly yang siap dirender/disimpan.

    Raises:
        ValueError: Jika dataframe kosong atau kolom 'Segment' tidak ditemukan.
    """
    # Validasi awal integritas data (Fail-Fast)
    if df_segmented.empty or 'Segment' not in df_segmented.columns:
        raise ValueError("Pipeline Error: Dataframe kosong atau kolom 'Segment' tidak ditemukan.")

    logger.info("Memulai pembuatan visualisasi Plotly bergaya McKinsey Tier Corporate...")
    
    # Agregasi data frekuensi segmen
    segment_counts = df_segmented['Segment'].value_counts()
    categories = segment_counts.index.tolist()
    values = segment_counts.values.tolist()

    # Menentukan warna bar secara strategis
    max_val_idx = values.index(max(values))
    bar_colors = [C["blue"] if i == max_val_idx else C["slate"] for i in range(len(values))]

    # Inisialisasi Grafik Batang
    fig = go.Figure(data=[
        go.Bar(
            x=categories,
            y=values,
            marker_color=bar_colors,
            marker_line_width=0, 
            textposition='none',  # Menghilangkan teks angka di atas bar
            cliponaxis=False
        )
    ])

    # Penerapan Aturan Layout McKinsey (Clean, Despined, High-Contrast Typography)
    fig.update_layout(
        paper_bgcolor=C["bg"],
        plot_bgcolor=C["bg"],
        width=850,
        height=500,
        margin=dict(l=60, r=40, t=100, b=80),
        showlegend=False,
        
        # Tipografi Judul & Subjudul Terpusat
        title=dict(
            text=(
                "<b>Distribusi Segmen Pelanggan Reseller</b><br>"
                "<span style='font-size:13px; color:#6B7B8D; font-weight:normal;'>"
                "Analisis berbasis volume kuantitas untuk identifikasi kelompok kontributor utama"
                "</span>"
            ),
            font=dict(family="Arial", size=20, color=C["navy"]),
            x=0.07,
            y=0.92
        ),
        
        # Konfigurasi Sumbu X
        xaxis=dict(
            title=dict(
                text="Segmen Pelanggan", 
                font=dict(family="Arial", size=13, color=C["navy"]),
                standoff=15
            ),
            tickfont=dict(family="Arial", size=11, color=C["tick"]),
            showgrid=False,
            showline=True,
            linecolor=C["axis_line"],
            linewidth=1,
            mirror=False
        ),
        
        # Konfigurasi Sumbu Y
        yaxis=dict(
            title=dict(
                text="Jumlah Pelanggan", 
                font=dict(family="Arial", size=13, color=C["navy"]),
                standoff=15
            ),
            tickfont=dict(family="Arial", size=11, color=C["tick"]),
            showgrid=False,   # Mematikan grid background harian
            showline=False,
            zeroline=False    # Mematikan garis nol dasar sumbu Y
        ),
        
        # Penambahan Footnote/Keterangan Sumber
        annotations=[
            dict(
                x=0,
                y=-0.22,
                xref="paper",
                yref="paper",
                text="Sumber: Internal Data Lake Pipeline v1.0. Metrik dihitung menggunakan segmentasi kuintil 3-skala.",
                showarrow=False,
                font=dict(family="Arial", size=10, color=C["footnote"]),
                align="left"
            )
        ]
    )

    return fig


def save_plotly_as_png(figure: go.Figure, output_dir: str, filename: str) -> None:
    """Menyimpan berkas visualisasi Plotly sebagai gambar statis PNG beresolusi tinggi."""
    try:
        os.makedirs(output_dir, exist_ok=True)
        full_export_path = os.path.join(output_dir, filename)
        
        # PERBAIKAN: Menggunakan write_image dengan skala kuadrat (scale=3) untuk hasil cetak tajam 300 DPI
        figure.write_image(full_export_path, format="png", scale=3)
        logger.info(f"Artifact Gambar Sukses Diekspor ke: {full_export_path}")
    except PermissionError:
        logger.error(
            f"I/O Error: Akses ditolak saat menulis gambar ke {filename}. "
            "Periksa kembali hak akses direktori atau pastikan file tidak terkunci."
        )
    except Exception as e:
        logger.error(f"Gagal mengekspor berkas PNG Plotly: {str(e)}")

# =============================================================================
# 2. PIPELINE EXECUTION GATEWAY
# =============================================================================
try:
    print("=== VISUALISASI PLOTLY (MCKINSEY CORPORATE TIER - PNG EXPORT) ===")

    # Eksekusi pembuatan chart
    fig_corporate = generate_corporate_segment_chart(df_segmented=rfm_final)

    # Simpan hasil ekspor komponen grafik ke dalam format gambar PNG secara defensif
    save_plotly_as_png(
        figure=fig_corporate,
        output_dir=OUTPUT_CHARTS_DIR,
        filename=OUTPUT_FILENAME_PNG
    )

    # Render visualisasi ke layar VS Code Notebook
    fig_corporate.show()

except Exception as e:
    logger.error(f"Komponen Visualisasi Korporat Plotly Gagal: {str(e)}")
    raise e

=== VISUALISASI PLOTLY (MCKINSEY CORPORATE TIER - PNG EXPORT) ===
2026-06-22 21:40:24,967 [INFO] Memulai pembuatan visualisasi Plotly bergaya McKinsey Tier Corporate...


2026-06-22 21:40:25,640 [INFO] Chromium init'ed with kwargs {}
2026-06-22 21:40:25,646 [INFO] Found chromium path: C:\Program Files\Google\Chrome\Application\chrome.exe
2026-06-22 21:40:25,650 [INFO] Temp directory created: C:\Users\Vertin\AppData\Local\Temp\tmpdnt5q4xf.
2026-06-22 21:40:25,655 [INFO] Opening browser.
2026-06-22 21:40:25,659 [INFO] Temp directory created: C:\Users\Vertin\AppData\Local\Temp\tmpc018pxlz.
2026-06-22 21:40:25,665 [INFO] Temporary directory at: C:\Users\Vertin\AppData\Local\Temp\tmpc018pxlz
2026-06-22 21:40:27,047 [INFO] Conforming 1 to file:///C:/Users/Vertin/AppData/Local/Temp/tmpdnt5q4xf/index.html
2026-06-22 21:40:30,342 [INFO] Getting tab from queue (has 1)
2026-06-22 21:40:30,344 [INFO] Got 4F43
2026-06-22 21:40:30,585 [INFO] Reloading tab 4F43 before return.
2026-06-22 21:40:31,060 [INFO] Putting tab 4F43 back (queue size: 0).
2026-06-22 21:40:31,063 [INFO] Waiting for all cleanups to finish.
2026-06-22 21:40:31,065 [INFO] Exiting Kaleido.
2026-06-22

### 1. Temuan Utama dan Distribusi Segmen

Grafik menunjukkan total sampel sebanyak **12 pelanggan** yang terbagi ke dalam **3 segmen utama** berdasarkan metrik kuantitas (volume pembelian):

- **Potential Loyalist (6 pelanggan)**  
  Segmen terbesar yang mendominasi bagan (**50% dari total pelanggan**). Kelompok ini memiliki frekuensi atau volume pembelian yang cukup baik dan menunjukkan potensi besar untuk dikonversi menjadi pelanggan loyal.

- **At Risk (4 pelanggan)**  
  Segmen terbesar kedua (**±33% dari total pelanggan**). Mereka mulai menunjukkan penurunan aktivitas pembelian atau frekuensi transaksi, sehingga berisiko beralih ke kompetitor apabila tidak segera dilakukan intervensi.

- **Loyal (2 pelanggan)**  
  Segmen terkecil secara jumlah (**±17% dari total pelanggan**), namun umumnya menjadi kontributor penting terhadap stabilitas pendapatan karena loyalitas dan volume pembelian yang konsisten tinggi.

---

### 2. Implikasi Strategis dan Rekomendasi Promosi

Untuk memaksimalkan penjualan reseller, strategi pemasaran perlu disesuaikan dengan karakteristik masing-masing segmen pelanggan.

| Segmen Pelanggan | Karakteristik | Rekomendasi Strategi Promosi |
|------------------|---------------|------------------------------|
| **Potential Loyalist**<br>(6 pelanggan) | Pembelian aktif dengan volume moderat hingga tinggi. Membutuhkan dorongan tambahan untuk berkembang menjadi pelanggan loyal. | **Program Upselling & Insentif Kuantitas**<br><br>• Terapkan promo bertingkat (*tiered promotion*), misalnya: *"Beli 10 pack Botol PET, dapat diskon 10% untuk pembelian berikutnya."*<br>• Berikan akses awal terhadap produk baru atau sampel gratis produk komplementer, seperti varian tutup botol terbaru. |
| **At Risk**<br>(4 pelanggan) | Menunjukkan penurunan aktivitas pembelian dan frekuensi transaksi yang mulai melambat. | **Kampanye Re-engagement & Win-Back**<br><br>• Kirimkan voucher khusus *"Kami Merindukan Anda"* dengan masa berlaku terbatas (3–5 hari).<br>• Lakukan survei kepuasan singkat melalui WhatsApp atau chat otomatis untuk mengidentifikasi penyebab penurunan transaksi, seperti harga, ketersediaan stok, atau kendala pengiriman. |
| **Loyal**<br>(2 pelanggan) | Pelanggan inti dengan frekuensi pembelian tinggi dan tingkat kepercayaan yang kuat terhadap produk reseller. | **Program Retensi Eksklusif & Advocacy**<br><br>• Masukkan ke dalam program *VIP Seller* dengan harga grosir khusus dan persyaratan minimum order yang lebih fleksibel.<br>• Berikan prioritas pengiriman (*fast-track shipping*) serta jaminan alokasi stok ketika terjadi keterbatasan persediaan. |

#### Prioritas Eksekusi

1. **Fokus utama:** Konversi segmen **Potential Loyalist** menjadi pelanggan **Loyal** karena jumlahnya paling besar dan peluang peningkatan pendapatannya tinggi.
2. **Fokus kedua:** Jalankan program **win-back** untuk segmen **At Risk** guna mencegah kehilangan pelanggan yang sebelumnya sudah pernah bertransaksi.
3. **Fokus ketiga:** Pertahankan kepuasan segmen **Loyal** melalui program eksklusif agar mereka tetap aktif dan berpotensi menjadi advokat bisnis.